## Load packages

In [1]:
import sys, traceback
from plantcv import plantcv as pcv
import cv2
from cv2 import aruco

import numpy as np
import glob
import os
import pandas as pd
import timeit

print("Packages loaded successfully!")

Packages loaded successfully!


## Inputs chunk

Just change the directories below:

In [2]:
inputdir = "C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/TeparyBean_Phenocage_2022_2023/Ex8_Phenocage_July2023/071723/Pot_without_ar_code"
outdir = 'C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/TeparyBean_Phenocage_2022_2023/Ex8_Phenocage_July2023/071723/Pot_without_ar_code/Processed/'
resultsdir = "C:/Users/Aparna/Box/Plant Architecture Lab/Aparna/TeparyBean_Main/TeparyBean_Phenocage_2022_2023/Ex8_Phenocage_July2023/071723/Pot_without_ar_code/Results/"

foldername = os.path.basename(os.path.normpath(resultsdir))

# # Change options here if you want to visualize or save images - only one possible at a time
# visualize = True
# if(visualize):
#     param_var = "plot"
#     print("The steps will be plotted, but NOT saved!")
# else:
#     param_var = "print"
#     print("The steps will NOT be plotted, but saved in " + str(foldername) + " folder!")


# Create an empty dataframe to store values from images
results_df = pd.DataFrame(columns = ['Image_name', 'Area', 'Pot_no'])
# print(df)


## Processing chunk

In [3]:
# debug: Debugging mode. Values are None, "print", or "plot". Default = None.

# if (visualize):
#     pcv.params.debug = param_var
# else:    
#     pcv.params.debug = "None"

pcv.params.debug = "None"
# print(param_var)

os.getcwd()
os.chdir(inputdir)
image_list = glob.glob('*.jpg')
print("Number of images = ", len(image_list))

# Create an empty dataframe to store values from images
results_df = pd.DataFrame(columns = ['Image_name', 'Area'])
# print(results_df)

start = timeit.default_timer()    

for i in range(len(image_list)):

    my_picture = image_list[i]
    print("Processing...["+str(i+1)+ "/"+str(len(image_list))+"] --> "+ str(my_picture))

    pcv.params.debug = "None"
    # STEP 1: Read image
    img, path, img_filename = pcv.readimage(filename=my_picture, mode="native")
    
    #=====================
    # Aruco marker detection

    # gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # aruco_dict = aruco.Dictionary_get(aruco.DICT_6X6_250)
    # parameters =  aruco.DetectorParameters_create()
    # corners, ids, rejectedImgPoints = aruco.detectMarkers(gray, aruco_dict, parameters=parameters)
    # print("Pot no = " +str(ids[0,0]))

    #=====================
    
    pcv.params.debug = "None"

    # Corrects image based on color standard and stores output as corrected_img
    corrected_img = pcv.white_balance(img=img, mode='hist', roi=[1500, 1300, 100, 100])

    #=============
    # STEP 2: Convert image from RGB colorspace to LAB and HSV colorspace
    v = pcv.rgb2gray_hsv(rgb_img=corrected_img, channel='v')
    a = pcv.rgb2gray_lab(rgb_img=corrected_img, channel='a')

    #=============
    # STEP 3 Set a binary threshold on the saturation channel image
    # Threshold the a channel image
    v_thresh = pcv.threshold.binary(gray_img=v, threshold=110, max_value=111,object_type='dark')
    a_thresh1 = pcv.threshold.binary(gray_img=a, threshold=106, max_value=np.max(a), object_type='dark')

    # median filter on a_thresh image

    a_thresh = pcv.median_blur(a_thresh1, 2)

    pcv.params.debug = "None"
    #=============
    # STEP 4: Join the threshold from two layers of masking
    v_a = pcv.logical_or(bin_img1=v_thresh, bin_img2=a_thresh)

    #=============
    # STEP 5: Apply image mask based v_a combined channel
    masked = pcv.apply_mask(img=corrected_img , mask=v_a, mask_color='white')

    #=============
    # STEP 6 Identify objects
    # img - RGB or grayscale image data for plotting
    # mask - Binary mask used for detecting contours
    id_objects, obj_hierarchy = pcv.find_objects(img=masked, mask=a_thresh)

    #=============
    # STEP 7: Define region of interest (ROI)
    roi1, roi_hierarchy= pcv.roi.rectangle(img=masked, x=400, y=200, h=1160, w=1300)
    #=============
    # STEP 8: Keep objects that overlap with the ROI
    roi_objects, hierarchy3, kept_mask, obj_area = pcv.roi_objects(img=corrected_img, roi_contour=roi1,
                                                                roi_hierarchy=roi_hierarchy,
                                                                object_contour=id_objects,
                                                                obj_hierarchy=obj_hierarchy,
                                                                roi_type='partial')

    #=============
    # STEP 9 Object combine kept objects
    # Inputs:
    #   img - RGB or grayscale image data for plotting
    #   contours - Contour list
    #   hierarchy - Contour hierarchy array
    obj, mask = pcv.object_composition(img=corrected_img, contours=roi_objects, hierarchy=hierarchy3)

    img_copy = np.copy(corrected_img)

    #=============
    # Filter objects by ROI
    roi = roi1
    hierarchy = roi_hierarchy

    filtered_contours, filtered_hierarchy, filtered_mask, filtered_area = pcv.roi_objects(
        img=img_copy, roi_type="partial", roi_contour=roi,roi_hierarchy = hierarchy,object_contour=roi_objects, 
        obj_hierarchy=hierarchy3)

    # Combine objects together in each plant
    plant_contour, plant_mask = pcv.object_composition(img=img_copy, contours=filtered_contours, hierarchy=filtered_hierarchy)

    # Analyze the shape of each plant
    analysis_images = pcv.analyze_object(img=img_copy, obj=plant_contour, mask=plant_mask)
    analysis_images

    # Save the image with shape characteristics
    img_copy = analysis_images
    # obs = pcv.outputs.add_observation(sample='default', variable = 'roi', trait = 'roi', method = 'roi', scale = 'int', datatype = int, value= my_picture, label = '#')
    # print(pcv.outputs.observations)
    # print(pcv.outputs.observations['default']['perimeter']['value'])

    pcv.params.debug = "None"

    # Look at object area data without writing to a file 
    plant_area = pcv.outputs.observations['default']['area']['value']
    
    # print(plant_area)

    # if (visualize):
    #     pcv.params.debug = param_var
    # else:    
    #     pcv.params.debug = "print"

    outfile = os.path.join(outdir, my_picture)
    pcv.print_image(img=img_copy, filename=outfile)


    # Create the new row as its own dataframe
    row_df = pd.DataFrame({'Image_name' : [my_picture], 'Area' : [plant_area]})
    # row_df = pd.DataFrame({'Image_name' : [my_picture], 'Area' : [plant_area], 'Pot_no' : [ids[0,0]]})
    results_df = pd.concat([results_df, row_df])
    # print(df)

end = timeit.default_timer()

print("/n Time taken = ", (end-start), " sec.")
# print(df)


Number of images =  7
Processing...[1/7] --> 2023.07.17-14.59.16.RaspiZ_side1.jpg
Processing...[2/7] --> 2023.07.17-14.59.16.RaspiZ_side2.jpg
Processing...[3/7] --> 2023.07.17-14.59.16.RaspiZ_side3.jpg
Processing...[4/7] --> 2023.07.17-14.59.16.RaspiZ_side4.jpg
Processing...[5/7] --> 2023.07.17-14.59.16.RaspiZ_side5.jpg
Processing...[6/7] --> 2023.07.17-14.59.16.RaspiZ_side6.jpg
Processing...[7/7] --> 2023.07.17-14.59.16.RaspiZ_side7.jpg
/n Time taken =  26.333028699998977  sec.


## Save results to folder

In [4]:
results_df.to_csv(resultsdir+"Result_Ex8_071723_W.csv", index=False)
print("Results successfully saved in * " + str(foldername) + " * folder!")


Results successfully saved in * Results * folder!
